In [ ]:
# ── 1. LOAD THE DATA ──────────────────────────────────────────────────────────
# The file uses latin-1 encoding (not UTF-8) because of special characters
# Each line looks like: "Company profits rose sharply.@positive"
# We split on the LAST @ symbol to separate sentence from label

import pandas as pd

with open('Sentences_75Agree.txt', 'r', encoding='latin-1') as f:
    lines = f.readlines()

# Split each line into sentence and label
data = [line.strip().rsplit('@', 1) for line in lines if '@' in line]

# Convert to a dataframe
df = pd.DataFrame(data, columns=['sentence', 'label'])

# Check it loaded correctly
print(df.shape)
print(df['label'].value_counts())
print(df.head())

In [ ]:
# ── 2. SPLIT THE DATA ─────────────────────────────────────────────────────────
# We split into 80% training and 20% test
# stratify=df['label'] ensures each split has the same class proportions
# This is important given the class imbalance we can see above

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['sentence'],      # input text
    df['label'],         # labels (positive/neutral/negative)
    test_size=0.2,       # 20% held out for testing
    random_state=42,     # makes results reproducible
    stratify=df['label'] # preserves class balance in each split
)

print(f"Training set: {len(X_train)} sentences")
print(f"Test set:     {len(X_test)} sentences")
print(f"\nTraining label distribution:")
print(y_train.value_counts())

In [ ]:
# ── 3. MODEL 1: TF-IDF + LOGISTIC REGRESSION ─────────────────────────────────
# TF-IDF converts text into numbers based on word frequency
# Words that are frequent within a sentence but rare across the corpus receive higher weights
# We then use GridSearchCV to find the best value of C (regularisation strength)
# class_weight='balanced' accounts for the fact we have fewer negative examples

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, f1_score

# Step 1: Convert text to TF-IDF vectors
# stop_words='english' removes common words like "the", "and" etc.
tfidf = TfidfVectorizer(stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)  # fit on training data only
X_test_tfidf  = tfidf.transform(X_test)        # apply same transformation to test

# Step 2: GridSearch over C values to find the best regularisation strength
param_grid = {'C': [0.01, 0.1, 1, 10, 100]}

grid_tfidf = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    param_grid,
    scoring='f1_macro',  # macro F1 is our headline metric given class imbalance
    cv=5,                # 5-fold cross validation
    verbose=1
)

grid_tfidf.fit(X_train_tfidf, y_train)

print(f"Best C value: {grid_tfidf.best_params_}")
print(f"Best cross-validation macro F1: {grid_tfidf.best_score_:.4f}")

# Step 3: Evaluate on the test set
y_pred_tfidf = grid_tfidf.predict(X_test_tfidf)

print("\nTest Set Performance — TF-IDF + Logistic Regression")
print("=" * 55)
print(f"Macro F1: {f1_score(y_test, y_pred_tfidf, average='macro'):.4f}")
print("\nPer-class breakdown:")
print(classification_report(y_test, y_pred_tfidf))

In [ ]:
# ── 4. MODEL 2: SENTENCE EMBEDDINGS + LOGISTIC REGRESSION ────────────────────
# Instead of counting words (TF-IDF), we use a pre-trained transformer model
# to convert each sentence into a list of 384 numbers that capture its meaning
# Sentences with similar meanings will have similar numbers
# This should help with financial language where meaning matters more than words

from sentence_transformers import SentenceTransformer

# Step 1: Load the pre-trained embedding model
# all-MiniLM-L6-v2 is fast, small, and works well for sentence-level tasks
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Step 2: Convert sentences to embeddings
# show_progress_bar=True lets us see how far along it is
print("Encoding training sentences...")
X_train_emb = embedding_model.encode(X_train.tolist(), show_progress_bar=True)

print("Encoding test sentences...")
X_test_emb = embedding_model.encode(X_test.tolist(), show_progress_bar=True)

print(f"\nEmbedding shape: {X_train_emb.shape}")
print("(rows = sentences, columns = embedding dimensions)")

In [ ]:
# ── 5. TRAIN AND EVALUATE MODEL 2 ────────────────────────────────────────────
# Now we run the same GridSearch as Model 1 but using embeddings instead of TF-IDF
# The logistic regression is identical — the only difference is the input features
# This makes the comparison clean and meaningful

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

# GridSearch over C values — same as Model 1 for a fair comparison
param_grid = {'C': [0.01, 0.1, 1, 10, 100]}

grid_emb = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    param_grid,
    scoring='f1_macro',  # same metric as Model 1
    cv=5,                # same number of folds as Model 1
    verbose=1
)

grid_emb.fit(X_train_emb, y_train)

print(f"Best C value: {grid_emb.best_params_}")
print(f"Best cross-validation macro F1: {grid_emb.best_score_:.4f}")

# Evaluate on test set
y_pred_emb = grid_emb.predict(X_test_emb)

print("\nTest Set Performance — Embeddings + Logistic Regression")
print("=" * 55)
print(f"Macro F1: {f1_score(y_test, y_pred_emb, average='macro'):.4f}")
print("\nPer-class breakdown:")
print(classification_report(y_test, y_pred_emb))

In [ ]:
# ── 6. ERROR ANALYSIS ─────────────────────────────────────────────────────────
# We look at sentences the WINNING model (embeddings) got wrong
# This helps us understand WHY the model makes mistakes
# In financial text, errors often reveal where specialist language causes confusion

import pandas as pd

# Create a dataframe with the test sentences, true labels and predicted labels
error_df = pd.DataFrame({
    'sentence': X_test.values,
    'true_label': y_test.values,
    'predicted_label': y_pred_emb
})

# Filter to only the misclassified sentences
errors = error_df[error_df['true_label'] != error_df['predicted_label']]

print(f"Total misclassified: {len(errors)} out of {len(error_df)} ({100*len(errors)/len(error_df):.1f}%)")
print(f"\nBreakdown of error types (true → predicted):")
print(errors.groupby(['true_label', 'predicted_label']).size().reset_index(name='count'))

print("\n--- Sample misclassified sentences ---")

# Show 5 examples for each error type
for true_label in ['negative', 'neutral', 'positive']:
    subset = errors[errors['true_label'] == true_label]
    print(f"\n[TRUE: {true_label.upper()}] — model got these wrong:")
    for _, row in subset.sample(n=min(3, len(subset)), random_state=42).iterrows():
        print(f"  Predicted: {row['predicted_label']}")
        print(f"  Sentence:  {row['sentence']}")
        print()

In [ ]:
# ── 7. MODEL COMPARISON TABLE ─────────────────────────────────────────────────
# A clean summary table comparing both models side by side
# This is what we'll reference in the write-up when justifying our final model choice

from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

# Calculate metrics for both models
results = pd.DataFrame({
    'Metric': ['Accuracy', 'Macro F1', 'Macro Precision', 'Macro Recall',
               'Negative F1', 'Neutral F1', 'Positive F1'],
    'TF-IDF + LogReg': [
        accuracy_score(y_test, y_pred_tfidf),
        f1_score(y_test, y_pred_tfidf, average='macro'),
        precision_score(y_test, y_pred_tfidf, average='macro'),
        recall_score(y_test, y_pred_tfidf, average='macro'),
        f1_score(y_test, y_pred_tfidf, labels=['negative'], average='macro'),
        f1_score(y_test, y_pred_tfidf, labels=['neutral'], average='macro'),
        f1_score(y_test, y_pred_tfidf, labels=['positive'], average='macro'),
    ],
    'Embeddings + LogReg': [
        accuracy_score(y_test, y_pred_emb),
        f1_score(y_test, y_pred_emb, average='macro'),
        precision_score(y_test, y_pred_emb, average='macro'),
        recall_score(y_test, y_pred_emb, average='macro'),
        f1_score(y_test, y_pred_emb, labels=['negative'], average='macro'),
        f1_score(y_test, y_pred_emb, labels=['neutral'], average='macro'),
        f1_score(y_test, y_pred_emb, labels=['positive'], average='macro'),
    ]
})

# Round to 4 decimal places
results['TF-IDF + LogReg'] = results['TF-IDF + LogReg'].round(4)
results['Embeddings + LogReg'] = results['Embeddings + LogReg'].round(4)

print("Model Comparison Table")
print("=" * 55)
print(results.to_string(index=False))